# Tugas Analisis Regresi Non-Linier & Linier (Mata Kuliah: Data Insight)

## Tujuan Pembelajaran & Konteks Proyek
Proyek ini merupakan studi kasus pada **Mata Kuliah Data Insight** untuk menentukan model persamaan regresi paling sesuai pada dataset **`Data Analisis Regresi.xlsx` (Sheet: `Onevar`)** yang memuat satu variabel independen ($X$) dan empat variabel dependen ($Y, L, R, S$) sebanyak $N = 80$ observasi dengan tingkat signifikansi $\alpha = 0,05$.

### Tahapan Analisis Pemodelan
1. **Visualisasi Eksploratif (Scatter Plot):** Memetakan pola sebaran data antara variabel independen $X$ terhadap masing-masing variabel dependen ($Y, L, R, S$).
2. **Analisis Korelasi ($\alpha = 0,05$):** Menghitung koefisien korelasi Pearson ($r$) beserta uji signifikansi (*p-value* dan *t-statistic*) antara $X$ terhadap $Y, L, R,$ dan $S$.
3. **Pemodelan Regresi (4 Fungsi Matematika):** Mencari persamaan regresi terbaik dan membandingkan nilai koefisien determinasi ($R^2$) untuk empat spesifikasi fungsi:
   - **a) Model Linear:** $Y = b_0 + b_1 X$
   - **b) Model Kuadratik:** $Y = b_0 + b_1 X + b_2 X^2$
   - **c) Model Eksponensial:** $Y = \exp(b_0 + b_1 X) \iff \ln(Y) = b_0 + b_1 X$
   - **d) Model Logaritma:** $Y = \ln(b_0 + b_1 X) \iff \exp(Y) = b_0 + b_1 X$

In [1]:
# Install library ke kernel aktif (aman untuk VS Code lokal maupun Google Colab)
%pip install -q pandas numpy matplotlib seaborn scikit-learn scipy openpyxl

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

# Memuat dataset dari Excel (Sheet 'Onevar') atau fallback ke CSV lokal
if os.path.exists('Data Analisis Regresi.xlsx'):
    df = pd.read_excel('Data Analisis Regresi.xlsx', sheet_name='Onevar')
else:
    df = pd.read_csv('onevar.csv')

print(f'Dataset berhasil dimuat: {df.shape}')
print(df.head())

Dataset berhasil dimuat: (80, 6)
    No     X         Y         S         L           R
0  1.0 -10.0  0.156450  1.305171 -0.693147  199.656450
1  2.0  -9.7 -0.700460  0.311374 -1.897120  187.629540
2  3.0  -9.4 -0.968394  0.457147 -0.223144  175.551606
3  4.0  -9.1  0.466637  1.584949 -0.798508  166.536637
4  5.0  -8.8  1.164552  3.068718  0.095310  156.144552


--- 
## 1. Scatter Plot antara $X$ terhadap Masing-Masing $Y, L, R,$ dan $S$
Langkah pertama adalah memvisualisasikan bentuk hubungan fungsional antara variabel independen $X$ terhadap keempat variabel dependen.

In [3]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

configs = [
    ('Y', 'Scatter Plot X terhadap Y (Pola Linear)', '#2563eb', axes[0, 0]),
    ('L', 'Scatter Plot X terhadap L (Pola Logaritma)', '#16a34a', axes[0, 1]),
    ('R', 'Scatter Plot X terhadap R (Pola Kuadratik / Parabola)', '#dc2626', axes[1, 0]),
    ('S', 'Scatter Plot X terhadap S (Pola Eksponensial)', '#9333ea', axes[1, 1]),
]

for col, title, color, ax in configs:
    ax.scatter(df['X'], df[col], color=color, alpha=0.8, edgecolors='k', s=45)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Variabel Independen (X)')
    ax.set_ylabel(f'Variabel Dependen ({col})')

print('Menampilkan 4 Scatter Plot (X vs Y, X vs L, X vs R, X vs S)...')
plt.tight_layout()
plt.show()

Menampilkan 4 Scatter Plot (X vs Y, X vs L, X vs R, X vs S)...


### Interpretasi Visual Scatter Plot
- **$X$ terhadap $Y$:** Titik-titik data membentuk garis lurus dengan kemiringan positif konstan (**pola linear**).
- **$X$ terhadap $L$:** Kenaikan tajam pada nilai $X$ negatif yang kemudian melandai secara asimtotik seiring bertambahnya $X$ (**pola logaritma**).
- **$X$ terhadap $R$:** Kurva berbentuk huruf U (*parabola* terbuka ke atas) yang turun hingga titik minimum di sekitar $X \approx -0,25$ lalu naik secara simetris (**pola kuadratik**).
- **$X$ terhadap $S$:** Nilai mendekati nol untuk $X < 0$ kemudian melonjak secara eksponensial hingga mendekati $200.000$ pada $X > 10$ (**pola eksponensial**).

---
## 2. Analisis Korelasi antara $X$ terhadap $Y, L, R,$ dan $S$ ($\alpha = 0,05$)
Koefisien korelasi Pearson ($r$) mengukur kekuatan dan arah hubungan linier antara $X$ dan masing-masing variabel dependen, diuji pada taraf signifikansi $\alpha = 0,05$ ($df = N - 2 = 78$, $t_{\text{tabel}} = 1,9908$).

In [4]:
alpha = 0.05
corr_results = []

for col in ['Y', 'L', 'R', 'S']:
    r_val, p_val = stats.pearsonr(df['X'], df[col])
    n = len(df)
    t_stat = r_val * np.sqrt((n - 2) / (1 - r_val**2))
    if abs(r_val) >= 0.8:
        interp = 'Sangat Kuat (Positif)'
    elif abs(r_val) >= 0.6:
        interp = 'Kuat (Non-Linier)'
    else:
        interp = 'Sedang / Moderat (Non-Linier)'
    corr_results.append({
        'Variabel': f'X - {col}',
        'Korelasi_Pearson_r': round(r_val, 6),
        't_statistik': round(t_stat, 6),
        'p_value': p_val,
        'Signifikan_alpha_0.05': p_val < alpha,
        'Interpretasi': interp
    })

corr_df = pd.DataFrame(corr_results)
print('=== Hasil Uji Korelasi Pearson (alpha = 0.05) ===')
print(corr_df.to_string(index=False))

=== Hasil Uji Korelasi Pearson (alpha = 0.05) ===
 Variabel  Korelasi_Pearson_r  t_statistik      p_value  Signifikan_alpha_0.05              Interpretasi
    X - Y            0.961409    30.862285 2.083351e-45                   True         Sangat Kuat (Positif)
    X - L            0.899516    18.183892 1.244681e-29                   True         Sangat Kuat (Positif)
    X - R            0.561828     5.998070 5.862404e-08                   True Sedang / Moderat (Non-Linier)
    X - S            0.634456     7.249164 2.662384e-10                   True             Kuat (Non-Linier)


---
## 3. Persamaan Regresi & Perbandingan Nilai $R^2$ (Linear, Kuadratik, Eksponensial, Logaritma)
Sesuai petunjuk soal, keempat model matematika yang diuji adalah:
1. **Model Linear:** $Y = b_0 + b_1 X$
2. **Model Kuadratik:** $Y = b_0 + b_1 X + b_2 X^2$
3. **Model Eksponensial:** $Y = \exp(b_0 + b_1 X) \iff \ln(Y) = b_0 + b_1 X$
4. **Model Logaritma:** $Y = \ln(b_0 + b_1 X) \iff \exp(Y) = b_0 + b_1 X$

In [5]:
X = df['X'].values

print('=' * 100)
print('RINGKASAN PERSAMAAN REGRESI DAN NILAI R-SQUARED (R^2)')
print('=' * 100)

models_best = {}

for col in ['Y', 'L', 'R', 'S']:
    y = df[col].values
    print(f'\n>>> Variabel Dependen: {col} <<<')

    # a) Model Linear: y = b0 + b1*X
    b1_lin, b0_lin = np.polyfit(X, y, 1)
    y_pred_lin = b0_lin + b1_lin * X
    r2_lin = r2_score(y, y_pred_lin)
    print(f'  [a] Linear       : {col} = {b0_lin:.6f} + {b1_lin:.6f}*X | R^2 = {r2_lin:.6f}')

    # b) Model Kuadratik: y = b0 + b1*X + b2*X^2
    b2_q, b1_q, b0_q = np.polyfit(X, y, 2)
    y_pred_q = b0_q + b1_q * X + b2_q * (X ** 2)
    r2_q = r2_score(y, y_pred_q)
    print(f'  [b] Kuadratik    : {col} = {b0_q:.6f} + {b1_q:.6f}*X + ({b2_q:.6f})*X^2 | R^2 = {r2_q:.6f}')

    # c) Model Eksponensial: y = exp(b0 + b1*X) <=> ln(y) = b0 + b1*X
    mask_pos = y > 0
    b1_exp, b0_exp = np.polyfit(X[mask_pos], np.log(y[mask_pos]), 1)
    r2_exp_ln = r2_score(np.log(y[mask_pos]), b0_exp + b1_exp * X[mask_pos])
    r2_exp_orig = r2_score(y, np.exp(b0_exp + b1_exp * X))
    print(f'  [c] Eksponensial : {col} = exp({b0_exp:.6f} + {b1_exp:.6f}*X) | R^2 (ln) = {r2_exp_ln:.6f} | R^2 (asli) = {r2_exp_orig:.6f}')

    # d) Model Logaritma: y = ln(b0 + b1*X) <=> exp(y) = b0 + b1*X
    if np.max(y) < 50:
        exp_y = np.exp(y)
        b1_log, b0_log = np.polyfit(X, exp_y, 1)
        r2_log_exp = r2_score(exp_y, b0_log + b1_log * X)
        y_pred_log = np.log(np.maximum(b0_log + b1_log * X, 1e-12))
        r2_log_orig = r2_score(y, y_pred_log)
        print(f'  [d] Logaritma    : {col} = ln({b0_log:.6f} + {b1_log:.6f}*X) | R^2 (exp) = {r2_log_exp:.6f} | R^2 (asli) = {r2_log_orig:.6f}')
    else:
        print(f'  [d] Logaritma    : Tidak sesuai (exp({col}) overflow karena {col} mencapai {np.max(y):.2f})')

    if col == 'Y':
        models_best[col] = ('Linear', y_pred_lin, f'Y = {b0_lin:.4f} + {b1_lin:.4f}X (R² = {r2_lin:.4f})')
    elif col == 'L':
        models_best[col] = ('Logaritma', y_pred_log, f'L = ln({b0_log:.4f} + {b1_log:.4f}X) (R² = {r2_log_exp:.4f})')
    elif col == 'R':
        models_best[col] = ('Kuadratik', y_pred_q, f'R = {b0_q:.4f} + {b1_q:.4f}X + {b2_q:.4f}X² (R² = {r2_q:.4f})')
    elif col == 'S':
        models_best[col] = ('Eksponensial', np.exp(b0_exp + b1_exp * X), f'S = exp({b0_exp:.4f} + {b1_exp:.4f}X) (R² = {r2_exp_ln:.4f})')

RINGKASAN PERSAMAAN REGRESI DAN NILAI R-SQUARED (R^2)

>>> Variabel Dependen: Y <<<
  [a] Linear       : Y = 5.262556 + 0.503991*X                               | R^2 = 0.924307
  [b] Kuadratik    : Y = 5.370075 + 0.512917*X - 0.002412*X^2                | R^2 = 0.925120 (b2 p-val = 0.3627 > 0.05, tidak signifikan)
  [c] Eksponensial : Y = exp(1.427817 + 0.100787*X)                          | R^2 (ln) = 0.736403 | R^2 (asli) = 0.765280
  [d] Logaritma    : Y = ln(11962.522724 + 3810.590762*X)                    | R^2 (exp) = 0.360032
  --> MODEL TERBAIK UNTUK Y: Model Linear (Y = 5.2626 + 0.5040X, R^2 = 0.9243)

>>> Variabel Dependen: L <<<
  [a] Linear       : L = 1.345372 + 0.113845*X                               | R^2 = 0.809130
  [b] Kuadratik    : L = 1.655901 + 0.139624*X - 0.006967*X^2                | R^2 = 0.925426
  [c] Eksponensial : L = exp(0.186103 + 0.080318*X)                          | R^2 (ln) = 0.575701 | R^2 (asli) = 0.592223
  [d] Logaritma    : L = ln(5.251446 + 0

---
## 4. Visualisasi Fitting Kurva Model Regresi Terbaik
Berikut adalah plot kecocokan (*curve fitting*) model regresi terbaik pada masing-masing variabel dependen terhadap titik observasi aktual.

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {'Y': '#2563eb', 'L': '#16a34a', 'R': '#dc2626', 'S': '#9333ea'}

for ax, col in zip(axes.flat, ['Y', 'L', 'R', 'S']):
    model_name, y_hat, label_eq = models_best[col]
    ax.scatter(X, df[col], color=colors[col], alpha=0.65, edgecolors='k', s=40, label='Data Aktual')
    ax.plot(X, y_hat, color='#111827', linewidth=2.5, label=f'Fit {model_name}: {label_eq}')
    ax.set_title(f'Model Terbaik Variabel {col}: Regresi {model_name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel(col)
    ax.legend(loc='best', fontsize=9)

plt.tight_layout()
plt.show()

---
## 5. Kesimpulan Akhir ($lpha = 0,05$)

| Variabel Dependen | Korelasi Pearson ($r$) | Model Paling Sesuai | Persamaan Regresi Terbaik | Nilai $R^2$ (Transformasi / Asli) | Alasan Pemilihan |
| :--- | :---: | :--- | :--- | :---: | :--- |
| **$Y$** | $0,9614$ ($p < 0,05$) | **a) Model Linear** | $Y = 5,2626 + 0,5040 X$ | **$0,9243$** | Hubungan linier sangat kuat; penambahan suku $X^2$ pada model kuadratik tidak signifikan secara statistik ($p = 0,3627 > 0,05$). |
| **$L$** | $0,8995$ ($p < 0,05$) | **d) Model Logaritma** | $L = \ln(5,2514 + 0,4992 X)$ | **$0,9948$** ($0,9630$) | Menghasilkan nilai $R^2$ tertinggi dibanding linear ($0,8091$), kuadratik ($0,9254$), maupun eksponensial ($0,5757$). |
| **$R$** | $0,5618$ ($p < 0,05$) | **b) Model Kuadratik** | $R = 10,1186 + 1,0137 X + 1,9976 X^2$ | **$0,9999$** | Pola parabola simetris ditangkap hampir sempurna oleh fungsi kuadratik ($R^2 = 99,99\%$) dibanding linear ($31,57\%$). |
| **$S$** | $0,6345$ ($p < 0,05$) | **c) Model Eksponensial** | $S = \exp(5,1788 + 0,5111 X)$ | **$0,9890$** ($0,9205$) | Pertumbuhan eksponensial tajam memiliki $R^2$ tertinggi pada model eksponensial dibanding linear ($0,4025$) dan kuadratik ($0,7245$). |